# 技术文档 · 原始交易明细与购物篮

推荐系统从客户的购买历史中提取商品。本文件记录从原始 Excel 核对一张发票的方法，并区分明细行数、商品种数、购买件数和金额。示例使用客户 `17850` 的发票 `536365`。

在 Colab 选择 CPU，按顺序点击各代码单元格左侧的 ▶，即可用原始数据复核文中的数字。

工作簿来自 D. Chen（2015）的 UCI Online Retail 数据集，许可为 CC BY 4.0。金额单位为英镑。[数据来源](https://doi.org/10.24432/C5BW33)

## 工作簿位置与读取权限

先在 Google Drive 里确认项目文件夹下有 `data/raw/Online Retail.xlsx`。如果文件夹在「与我共享」，可以把它作为快捷方式添加到「我的云端硬盘」。在 Colab 运行下一格，按提示授权挂载 Drive。

代码会先检查项目约定的路径。如果出现输入框，打开 Colab 左栏的「文件」，在 `drive` 下找到 Excel 文件，从它的菜单选择「复制路径」，粘贴到输入框并按回车。输入框接收文件路径。使用本地 Jupyter 时，代码会查找仓库内的数据文件。

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    drive.mount("/content/drive")
    workbook = Path("/content/drive/MyDrive/CS5481_Project2_OnlineRetail/data/raw/Online Retail.xlsx")
    if not workbook.is_file():
        workbook = Path(input("请粘贴从 Colab「文件」面板复制的 Excel 路径：").strip())
else:
    current_dir = Path.cwd()
    for directory in (current_dir, *current_dir.parents):
        workbook = directory / "data/raw/Online Retail.xlsx"
        if workbook.is_file():
            break

if not workbook.is_file():
    raise FileNotFoundError("没有找到工作簿。请核对文件名、路径和当前账号的读取权限，再运行本格。")
print("本次读取的文件：", workbook.resolve())

## 数据版本与表结构

SHA-256 是根据文件全部内容算出的校验值。同名文件也可能有不同内容。第一段把实际值和研究计划中的值比较，匹配后才继续。第二段读取名为 `Online Retail` 的工作表，把三个编号列按文本保存，以便保留商品编码里的字母。读取 54 万多行可能需要稍候。

In [ ]:
from hashlib import sha256
import pandas as pd

expected_hash = "43465a06f2ccf7c8b5bd2892bc7defb52f97487934fe93b16ae4c3936424676d"
actual_hash = sha256(workbook.read_bytes()).hexdigest()
print("文件 SHA-256：", actual_hash)
if actual_hash != expected_hash:
    raise ValueError("当前文件与研究计划记录的版本不同。请选择未经编辑的原始工作簿，再从第一步运行。")

identifiers = {"InvoiceNo": "string", "StockCode": "string", "CustomerID": "string"}
sales = pd.read_excel(workbook, sheet_name="Online Retail", dtype=identifiers)
print(f"共 {len(sales):,} 行交易明细，{len(sales.columns)} 列。")
print("字段：", ", ".join(sales.columns))
assert sales.shape == (541909, 8), "请检查工作表的行数和列数。"

## 交易明细字段

原表中的一行记录某张发票上一个商品编码的交易数量和单价。同一张发票可以占多行，同一商品编码也可能重复出现。

| 列名 | 含义 |
|---|---|
| InvoiceNo | 发票编号。同一张发票上的各行共用这个编号。 |
| StockCode | 商品编码，可能包含字母，例如 `85123A`。 |
| Description | 商品的文字描述。 |
| Quantity | 本行记录的商品件数。原表含负数，清洗时需区分购买与退货等记录。 |
| InvoiceDate | 发票记录的日期和时间。 |
| UnitPrice | 单件价格，单位为英镑。 |
| CustomerID | 客户编号。原表中部分行缺少此值。 |
| Country | 客户所在的国家或地区。 |

下面用客户编号和发票编号筛选原表。输出保留该发票的每一行，方便逐行核对商品、数量和单价。

In [ ]:
invoice = sales.loc[
    (sales["CustomerID"] == "17850") & (sales["InvoiceNo"] == "536365")
].copy()
invoice

## 发票汇总口径

对这张发票，`len(invoice)` 统计明细行数，`nunique()` 统计不同商品编码的数量，`Quantity.sum()` 统计总件数。计算金额时，先把每行的数量乘以该行的单价，再对行金额求和。这样可以保留同一商品在不同明细行上的价格差异。

In [ ]:
line_amount = invoice["Quantity"] * invoice["UnitPrice"]
print("明细行数：", len(invoice))
print("不同商品种数：", invoice["StockCode"].nunique())
print("购买总件数：", invoice["Quantity"].sum())
print(f"第一行金额：£{line_amount.iloc[0]:.2f}")
print(f"整张发票金额：£{line_amount.sum():.2f}")

## 核验结果及其在推荐系统中的用途

商品 `85123A` 在第一行出现，6 件 × £2.55 = £15.30。整张发票共 7 行、7 种商品、40 件，总金额为 £139.12。

构建购买历史时，同一位客户的一张有效购买发票构成一个购物篮。按项目研究计划，应先去掉完全重复的原始行并筛出有效购买，再把同一张发票中同一商品的多行合并：商品在这个篮子里记一次购买事件，件数和金额分别累加。同一客户在另一张发票上再次购买它，则保留为另一次购买事件。